In [7]:
import os
from dotenv import load_dotenv
from getpass import getpass

import nest_asyncio

nest_asyncio.apply()
load_dotenv()

True

In [8]:
CO_API_KEY = os.environ['CO_API_KEY']
GEMINI_API_KEY = os.environ['GEMINI_API_KEY']
QDRANT_URL = os.environ['QDRANT_URL']
QDRANT_API_KEY = os.environ['QDRANT_API_KEY']

In [9]:
from llama_index.core.settings import Settings
from llama_index.llms.cohere import Cohere
from llama_index.embeddings.gemini import GeminiEmbedding

Settings.llm = Cohere(model="command-r-plus", api_key=CO_API_KEY)

Settings.embed_model = GeminiEmbedding(
    model_name="models/text-embedding-004",
    api_key=GEMINI_API_KEY,
    title='llms gemini')

# Ingestion Pipeline

- 🔄 **IngestionPipeline Overview**: Utilizes `Transformations` applied to input data, modifying data into nodes, which are returned or inserted to a vector database.

- 💾 **Caching Mechanism**: Each node+transformation pair is cached, enhancing efficiency for identical subsequent operations by utilizing cached results.


### Using an `IngestionPipeline`

First, let's read in some data. 

In [10]:
from llama_index.core import SimpleDirectoryReader

documents = SimpleDirectoryReader(
    input_files = ["../data/pg10763.txt"], 
    filename_as_id=True).load_data()

# Ingestion Pipeline with Document Management


 •  💾 **Caching in IngestionPipeline**: Hashes and stores each node + transformation combination to speed up future processes with identical data.

 •  📁 **Local Cache Management**: The input nodes list and transformation pair are cached in the pipeline. When we apply the same transformation to that list of nodes again, the output nodes are retrieved from the cache.

 •  📚 **Docstore Attachment**:  Enables document management in the ingestion pipeline, using `doc_id` or `node.ref_doc_id` for identification. Prevents running a transformation on the same document multiple times by using the document ID and the hash of the document content to manage duplicates.

 •  🗂️ **Duplicate Handling**:
  - Maintains a `doc_id` to `document_hash` map to identify duplicates.

  - Re-processes documents if the same `doc_id` is found with a changed hash.

  - Skips documents if the same `doc_id` is found but the hash remains unchanged.

 •  🚫 **Without Vector Store**:
  - Limited to checking and removing duplicate inputs.

 •  ✨ **With Vector Store**:
  - Enables handling of upserts for updated documents, offering advanced management capabilities.

In [11]:
from qdrant_client import QdrantClient

from llama_index.core import StorageContext, VectorStoreIndex
from llama_index.core.ingestion import IngestionCache, IngestionPipeline
from llama_index.core.node_parser import TokenTextSplitter
from llama_index.core.storage.docstore import SimpleDocumentStore
from llama_index.vector_stores.qdrant import QdrantVectorStore

client = QdrantClient(
    url=QDRANT_URL, 
    api_key=QDRANT_API_KEY,
)

vector_store = QdrantVectorStore(
    client=client, 
    collection_name="it_can_be_done")

ingest_cache = IngestionCache(
    collection="it_can_be_done",
)

# create pipeline with transformations
pipeline = IngestionPipeline(
    transformations=[
        TokenTextSplitter(chunk_size=256, chunk_overlap=16),
        Settings.embed_model
    ],
    docstore=SimpleDocumentStore(),
    vector_store=vector_store,
    cache=ingest_cache,
)

# run the pipeline
nodes = pipeline.run(documents = documents)

In [12]:
nodes[0].__dict__.keys()

dict_keys(['id_', 'embedding', 'metadata', 'excluded_embed_metadata_keys', 'excluded_llm_metadata_keys', 'relationships', 'text', 'mimetype', 'start_char_idx', 'end_char_idx', 'text_template', 'metadata_template', 'metadata_seperator'])

In [13]:
index = VectorStoreIndex.from_vector_store(vector_store=vector_store)

In [14]:
retriever = index.as_retriever(
    similarity_top_k=7, 
    return_sources=True
    )

In [15]:
retrieved_nodes = retriever.retrieve("Poems about starting where you stand, and not making dreams your master")

In [16]:
retrieved_nodes

[NodeWithScore(node=TextNode(id_='1f2c12ab-e0c9-4e80-a86b-86222be25abe', embedding=None, metadata={'file_path': '..\\data\\pg10763.txt', 'file_name': 'pg10763.txt', 'file_type': 'text/plain', 'file_size': 405245, 'creation_date': '2024-10-20', 'last_modified_date': '2024-10-20'}, excluded_embed_metadata_keys=['file_name', 'file_type', 'file_size', 'creation_date', 'last_modified_date', 'last_accessed_date'], excluded_llm_metadata_keys=['file_name', 'file_type', 'file_size', 'creation_date', 'last_modified_date', 'last_accessed_date'], relationships={<NodeRelationship.SOURCE: '1'>: RelatedNodeInfo(node_id='..\\data\\pg10763.txt', node_type=<ObjectType.DOCUMENT: '4'>, metadata={'file_path': '..\\data\\pg10763.txt', 'file_name': 'pg10763.txt', 'file_type': 'text/plain', 'file_size': 405245, 'creation_date': '2024-10-20', 'last_modified_date': '2024-10-20'}, hash='972e8b374665c7f9068d7e42a54213e462efb57aaa828d5a6d43ccbf167fa94e'), <NodeRelationship.PREVIOUS: '2'>: RelatedNodeInfo(node_id='

In [17]:
print(retrieved_nodes[0].get_text())

good, nor talk too wise:

  If you can dream--and not make dreams your master;
    If you can think--and not make thoughts your aim,
  If you can meet with Triumph and Disaster
    And treat those two imposters just the same;
  If you can bear to hear the truth you've spoken
    Twisted by knaves to make a trap for fools,
  Or watch the things you gave your life to, broken,
    And stoop and build 'em up with worn-out tools:

  If you can make one heap of all your winnings
    And risk it on one turn of pitch-and-toss,
  And lose, and start again at your beginnings
    And never breathe a word about your loss;
  If you can force your heart and nerve and sinew
    To serve your turn long after they are gone,
  And so hold on when there is nothing in you
    Except the Will which says to them; "Hold on!"

  If you can talk with crowds and keep your virtue,
    Or walk with Kings--nor lose the common


In [18]:
print(retrieved_nodes[0].get_score())

0.7253753


In [19]:
pipeline.persist('./pipeline_storage')